In [ ]:
import pyecap
import os
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.io as sio
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from flexNIRs.fnirs_functions import *

In [ ]:
meta_index = 2
metaDF = pd.read_excel(r'D:\Data\TCD\20260220_TCD_QC\meta.xlsx')
tank_folder = r'D:\Data\TCD\20260220_TCD_QC\CVP_SingleCh_Ramp-260220\\'

raw_ecg = pyecap.Ephys(tank_folder + metaDF.loc[meta_index, 'Tank'], stores = 'ECGG')
raw_ecg = raw_ecg.remove_ch(['ECGG 2','ECGG 3', 'ECGG 4'])

raw_tcd = pyecap.Ephys(tank_folder + metaDF.loc[meta_index, 'Tank'], stores = 'TCDD')

stim = pyecap.Stim(tank_folder + metaDF.loc[meta_index, 'Tank'])
stimDF = stim.parameters
stimDF

In [ ]:
ecg_d = raw_ecg.array[0,:].compute()
tcd_d = raw_tcd.array[0,:].compute()

ecg_f = tdt_ecg = filter_hr(raw_ecg, cutoff = 100, downsample=False, check_plot=False)[0] *1e-6
tcd_f = filter_hr(raw_tcd, cutoff = 100, downsample=False, check_plot=False)[0] *1e-6

time = raw_ecg.time()

In [ ]:
DS_factor = 20

tcd_DS = signal.decimate(tcd_d, DS_factor, zero_phase = True)
time_ds = np.arange(len(tcd_DS)) * (DS_factor / raw_tcd.sample_rate)

tcd_DS_f = filter_hr(raw_tcd, cutoff = 10, downsample=True, downsample_factor = DS_factor, check_plot=False)[0] *1e-6

#fig = go.Figure()
#fig.add_trace(go.Scattergl(x = time_ds, y = tcd_DS, name='Raw DS'))
#fig.add_trace(go.Scattergl(x = time_ds, y = tcd_DS_f, name='Filtered DS'))
#fig.show()

In [ ]:
dvt = np.gradient(tcd_DS_f, DS_factor / raw_tcd.sample_rate) ** 3
fig = make_subplots(specs = [[{'secondary_y' : True}]])
#fig.add_trace(go.Scattergl(x = time_ds, y = tcd_DS_f, name='Filtered DS'))
fig.add_trace(go.Scattergl(x = time_ds, y = tcd_DS, name='Raw DS'))
fig.add_trace(go.Scattergl(x = time_ds, y = dvt, name='1st Dvt'), secondary_y = True)
fig.show()